# CogVideoX inference server (free GPU on Google Colab)

This notebook exposes the open-source CogVideoX models as a small HTTP API so that
`cogvideox_client.py` on your machine can call it the same way it calls the Agnes video API.

**Two modes. A free Colab session can only run one at a time (not enough memory for both models).
Switch by setting `SERVER_MODE` in step 2, then re-run the whole notebook:**

- `SERVER_MODE = "text"`  -> [CogVideoX-2b](https://huggingface.co/THUDM/CogVideoX-2b), text-to-video.
- `SERVER_MODE = "image"` -> [CogVideoX-5b-I2V](https://huggingface.co/THUDM/CogVideoX-5b-I2V),
  takes one image as the first frame and generates a video. **First frame only: Agnes-style
  first+last frame control is not supported.** The model is more than twice as large, so on a
  free T4 it is slower and more likely to hit memory limits.

**Usage**
1. Menu `Runtime` -> `Change runtime type` -> choose the **T4 GPU** accelerator (free tier).
2. Run every cell from top to bottom. The first run of "Load the model" downloads the weights
   (about 10 GB in `text` mode, more in `image` mode), which takes a few minutes up to ten or more.
3. The last cell prints an `https://xxxx.ngrok-free.app` URL. Put it in your `.env` as
   `COGVIDEOX_BASE_URL`, and put the value you chose for `SERVER_API_KEY` into `COGVIDEOX_API_KEY`.
4. Keep this notebook running; your local code can only reach it while it is connected. Free Colab
   reclaims idle or long-running sessions. If it disconnects, re-run from the top (the URL may change,
   so update `.env` accordingly).

**Measured** (free T4 + quantization): one 5-second clip in `text` mode takes about 13-16 minutes.
After several consecutive runs GPU memory can fragment and trigger `CUDA out of memory`; if that
happens, fully restart the runtime (`Runtime` -> `Disconnect and delete runtime`, not just reconnect).

## 1. Install packages

In [ ]:
!pip install -q diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg pillow
!pip install -q fastapi uvicorn nest-asyncio pyngrok
!pip install -q bitsandbytes

# The preinstalled torchao version often does not match the API the latest diffusers expects
# (e.g. ImportError: cannot import name 'FqnToConfig' from 'torchao.quantization').
# We only use fp16 / bitsandbytes inference and do not need torchao, so removing it is the cleanest fix.
!pip uninstall -y torchao -q

## 2. Configuration: API key and ngrok token

Open the Secrets panel (key icon on the left of Colab) and add two secrets, using exactly the names
the next cell reads:

- `SERVER_API_KEY`: any string you choose to act as a password. Copy it unchanged into `COGVIDEOX_API_KEY` in your local `.env`.
- `NGROK_AUTH_TOKEN`: sign in at https://dashboard.ngrok.com/get-started/your-authtoken and copy the token
  (the free ngrok tier requires an account to create tunnels).

After saving, make sure the "Notebook access" toggle next to each secret is on, otherwise the next
cell will read `None`.

In [ ]:
from google.colab import userdata

# Read from the Colab Secrets panel instead of hard-coding values in the notebook.
# Colab asks whether this notebook may access each secret the first time; click Allow.
# If you saved the secrets under different names, change the two strings below.
SERVER_API_KEY = userdata.get("SERVER_API_KEY")
NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")

# "text"  -> CogVideoX-2b, text-to-video.
# "image" -> CogVideoX-5b-I2V, takes one image as the first frame (no last-frame control).
# A free Colab session can only run one model at a time; after changing this, re-run the whole notebook.
SERVER_MODE = "image"

MODEL_NAME = "THUDM/CogVideoX-5b-I2V" if SERVER_MODE == "image" else "THUDM/CogVideoX-2b"

## 3. Load the model (the first run downloads the weights; allow a few minutes)

In [ ]:
import gc
import os

# After many consecutive generations GPU memory fragments: there is free space in theory, but the
# allocator cannot find one contiguous block for a new tensor and raises CUDA out of memory
# (even when nvidia-smi shows free memory). This setting lets PyTorch's allocator reuse freed
# blocks more flexibly. It must be set before importing torch, because the allocator is
# initialized on first CUDA use and later changes have no effect.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from transformers import T5EncoderModel, BitsAndBytesConfig as TransformersBnbConfig
from diffusers import (
    CogVideoXPipeline,
    CogVideoXImageToVideoPipeline,
    CogVideoXTransformer3DModel,
    BitsAndBytesConfig as DiffusersBnbConfig,
)

assert torch.cuda.is_available(), "No GPU detected. Use Runtime -> Change runtime type, pick T4 GPU, and re-run."

# Loading the text encoder (T5-xxl, ~4.7B params) and the transformer (~2B in text mode, ~5B in
# image mode) entirely in fp16 easily exceeds the standard free Colab system RAM (~12 GB). CPU
# offload does not help: it only shuffles the model around in system RAM, it does not make it smaller.
#
# Image mode (5B transformer), measured: with both components quantized to 8-bit the weights alone
# take ~14.2 GB, while the T4 has only 14.56 GB of VRAM, leaving almost nothing for the activations
# needed during generation, so it hits OOM immediately. 4-bit (nf4) roughly halves the weight size
# again and leaves enough working memory.
# Text mode (2B transformer) would be fine with 8-bit, but both modes use 4-bit here: safe and
# faster to load.
quant_kwargs = dict(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)

# device_map is deliberately pinned to {"": 0} (whole model on GPU 0) instead of "auto".
# The text encoder and transformer are loaded separately, and "auto" only looks at how much GPU
# memory is free at that moment. Each load can look fine on its own while the combination pushes
# some layers of one of them onto the CPU. Running matrix math of this size on the CPU takes
# minutes per step, so a single video would take well over ten minutes.
text_encoder = T5EncoderModel.from_pretrained(
    MODEL_NAME, subfolder="text_encoder", torch_dtype=torch.float16,
    quantization_config=TransformersBnbConfig(**quant_kwargs),
    device_map={"": 0},
)
gc.collect()
torch.cuda.empty_cache()

transformer = CogVideoXTransformer3DModel.from_pretrained(
    MODEL_NAME, subfolder="transformer", torch_dtype=torch.float16,
    quantization_config=DiffusersBnbConfig(**quant_kwargs),
    device_map={"": 0},
)
gc.collect()
torch.cuda.empty_cache()

# Image mode needs CogVideoXImageToVideoPipeline (accepts an image= argument);
# text mode uses the regular CogVideoXPipeline (text only, no image argument).
PipelineClass = CogVideoXImageToVideoPipeline if SERVER_MODE == "image" else CogVideoXPipeline
pipe = PipelineClass.from_pretrained(
    MODEL_NAME, text_encoder=text_encoder, transformer=transformer,
    torch_dtype=torch.float16,
)
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()
pipe.vae.to("cuda")   # The VAE is not quantized and small (a few hundred MB): keep it on the GPU, no offload needed
gc.collect()
torch.cuda.empty_cache()
print(f"Model loaded ({SERVER_MODE} mode, {MODEL_NAME}, 4-bit quantization).")

## 4. Define the API server (create job / poll status / download file)

The response shape deliberately mirrors the Agnes video API (`video_id`, `status`, `metadata.url`),
so the local `cogvideox_client.py` needs almost no parsing changes to switch between this server and Agnes.

In [ ]:
import base64
import io
import threading
import time
import uuid
from pathlib import Path
from queue import Queue, Empty
from typing import Optional

import requests as http_requests
from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel
from PIL import Image
from diffusers.utils import export_to_video

OUTPUT_DIR = Path("/content/cogvideox_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PUBLIC_BASE_URL = ""  # Filled in by the next-but-one cell once the ngrok tunnel exists

# Resolution used by the official CogVideoX-5b-I2V examples; in image mode the input image is resized to it.
IMAGE_WIDTH, IMAGE_HEIGHT = 720, 480

tasks = {}         # video_id -> {"status", "progress", "error", "file"}
job_queue = Queue()

app = FastAPI()


class CreateVideoRequest(BaseModel):
    model: Optional[str] = None
    prompt: str
    mode: str = "text"
    seconds: str = "5"
    aspect_ratio: str = "16:9"
    seed: Optional[int] = None
    # Image mode: accepts either a data URI (data:image/...;base64,...) or a public URL.
    # Unlike Agnes' first_frame, which only takes a public URL, this self-hosted server can
    # receive base64 directly, so the image does not have to be uploaded somewhere first.
    first_frame: Optional[str] = None


def check_auth(authorization: Optional[str]):
    if authorization != f"Bearer {SERVER_API_KEY}":
        raise HTTPException(status_code=401, detail="invalid api key")


def seconds_to_frames(seconds_str: str) -> int:
    """CogVideoX runs at 8 fps and the frame count must be 4k+1; clamped to 9..49 (about 1-6 seconds)."""
    try:
        seconds = float(seconds_str)
    except (TypeError, ValueError):
        seconds = 5.0
    raw = round(seconds * 8)
    k = max(2, min(12, round((raw - 1) / 4)))
    return 4 * k + 1


def load_first_frame(value: str) -> Image.Image:
    """Load first_frame (data URI or public URL) as a PIL image resized to the model's resolution."""
    if value.startswith("data:"):
        _, b64data = value.split(",", 1)
        raw = base64.b64decode(b64data)
    else:
        resp = http_requests.get(value, timeout=60)
        resp.raise_for_status()
        raw = resp.content
    image = Image.open(io.BytesIO(raw)).convert("RGB")
    return image.resize((IMAGE_WIDTH, IMAGE_HEIGHT))


def worker_loop():
    while True:
        try:
            video_id, req = job_queue.get(timeout=1)
        except Empty:
            continue
        tasks[video_id]["status"] = "processing"
        result = None
        frames = None
        try:
            num_frames = seconds_to_frames(req.seconds)
            generator = None
            if req.seed is not None:
                generator = torch.Generator(device="cuda").manual_seed(int(req.seed))

            if SERVER_MODE == "image":
                if req.mode != "image" or not req.first_frame:
                    raise ValueError(
                        "This server is in image mode: mode must be \"image\" and first_frame is required"
                    )
                image = load_first_frame(req.first_frame)
                result = pipe(
                    prompt=req.prompt, image=image,
                    height=IMAGE_HEIGHT, width=IMAGE_WIDTH,
                    num_frames=num_frames, guidance_scale=6,
                    num_inference_steps=50, generator=generator,
                )
            else:
                if req.mode != "text":
                    raise ValueError(f"This server is in text mode and does not support mode={req.mode}")
                result = pipe(
                    prompt=req.prompt, num_frames=num_frames,
                    guidance_scale=6, num_inference_steps=50, generator=generator,
                )

            frames = result.frames[0]
            out_path = OUTPUT_DIR / f"{video_id}.mp4"
            export_to_video(frames, str(out_path), fps=8)
            tasks[video_id].update(status="completed", progress=100, file=str(out_path))
        except Exception as e:
            tasks[video_id].update(status="failed", error=str(e)[:500])
        finally:
            # After several consecutive jobs GPU memory fragments (even with free space the allocator
            # may not find a contiguous block and raises CUDA OOM). After each job, explicitly drop the
            # intermediate results and force a collection so the memory really goes back to the
            # allocator, reducing the chance that the next job hits OOM.
            del result, frames
            gc.collect()
            torch.cuda.empty_cache()


threading.Thread(target=worker_loop, daemon=True).start()


@app.post("/v1/videos")
def create_video(req: CreateVideoRequest, authorization: Optional[str] = Header(None)):
    check_auth(authorization)
    video_id = uuid.uuid4().hex
    tasks[video_id] = {"status": "queued", "progress": 0}
    job_queue.put((video_id, req))
    return {"video_id": video_id, "id": video_id, "task_id": video_id}


@app.get("/v1/videos/{video_id}")
def get_video_status(video_id: str, authorization: Optional[str] = Header(None)):
    check_auth(authorization)
    task = tasks.get(video_id)
    if not task:
        raise HTTPException(status_code=404, detail="video_id not found")
    out = {"status": task["status"], "progress": task.get("progress", 0)}
    if task["status"] == "completed":
        out["metadata"] = {"url": f"{PUBLIC_BASE_URL}/v1/videos/{video_id}/file"}
    if task["status"] == "failed":
        out["error"] = task.get("error")
    return out


@app.get("/v1/videos/{video_id}/file")
def get_video_file(video_id: str, authorization: Optional[str] = Header(None)):
    check_auth(authorization)
    task = tasks.get(video_id)
    if not task or task.get("status") != "completed":
        raise HTTPException(status_code=404, detail="video not ready")
    return FileResponse(task["file"], media_type="video/mp4")


print(f"API defined ({SERVER_MODE} mode). The next cell starts the ngrok tunnel and the server.")

## 5. Open the ngrok tunnel and start the server

This cell prints the public URL; put it in your `.env` as `COGVIDEOX_BASE_URL=<that URL>`.
The cell keeps running while the server is up; press the stop button on its left to shut it down.

In [ ]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

try:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
except Exception as e:
    # If ngrok detects the same token already in its config it prints "already set" instead of
    # "saved", which pyngrok's string matching treats as a failure. The token itself is fine.
    if "already set" not in str(e):
        raise
    print("(ngrok authtoken already configured, skipping.)")

# Do not pass bind_tls=True: that is a legacy ngrok v2 tunnel option. The current ngrok agent (v3)
# uses the new endpoint configuration format and rejects the field it would generate, returning
# 400 "field schemes not found in type config.Endpoint". v3 always uses TLS, so nothing is needed.
tunnel = ngrok.connect(8000)
PUBLIC_BASE_URL = tunnel.public_url

print("=" * 70)
print(f"COGVIDEOX_BASE_URL={PUBLIC_BASE_URL}")
print(f"COGVIDEOX_API_KEY={SERVER_API_KEY}")
print("Copy the two lines above into the .env file of the project that uses cogvideox_client.py.")
print("=" * 70)

# Do not use uvicorn.run(app, ...): it calls asyncio.run() internally, but this cell is already
# running inside an event loop, which raises
# RuntimeError: asyncio.run() cannot be called from a running event loop.
# Awaiting server.serve() attaches to the existing loop without going through asyncio.run().
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()